# Scan for selection (Fst), then view the sweep

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmdcolin/jbrowse-anywidget/blob/main/examples/06_popgen_selection.ipynb)

Same loop, population genetics: compute a windowed **Fst** between two populations and load it as a track to spot loci that differ. The synthetic data carries a selective sweep at *LCT/MCM6* — the lactase-persistence locus.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/cmdcolin/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## Simulate allele frequencies in two populations

Per-SNP alternate-allele frequencies over a stretch of chr2, with a sweep raising frequencies in population 1 around *LCT*. Swap for your own frequencies (e.g. from a multi-sample VCF).

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(1)
chrom, start, end = "2", 135_000_000, 137_000_000
pos = np.sort(rng.integers(start, end, size=4000))

anc = rng.uniform(0.05, 0.95, size=pos.size)  # shared ancestral freq
p1 = np.clip(anc + rng.normal(0, 0.05, pos.size), 0.001, 0.999)
p2 = np.clip(anc + rng.normal(0, 0.05, pos.size), 0.001, 0.999)

# selective sweep at LCT/MCM6: push pop-1 frequencies up
sweep = (pos >= 135_780_000) & (pos <= 135_860_000)
p1[sweep] = np.clip(p1[sweep] + 0.6, 0.001, 0.999)

## Windowed Hudson Fst

Fst per window as a ratio of summed numerators to denominators (the Bhatia et al. recommendation), in 20 kb windows.

In [ ]:
num = (p1 - p2) ** 2
den = p1 * (1 - p2) + p2 * (1 - p1)

win = 20_000
edges = np.arange(start, end + win, win)
w = np.searchsorted(edges, pos, side="right") - 1

rows = []
for k in range(len(edges) - 1):
    m = w == k
    if m.sum() >= 5:
        fst = float(num[m].sum() / den[m].sum())
        rows.append(
            {
                "chrom": chrom,
                "start": int(edges[k]),
                "end": int(edges[k + 1]),
                "fst": round(max(fst, 0.0), 3),
                "n_snps": int(m.sum()),
            }
        )

windows = pd.DataFrame(rows)
windows.sort_values("fst", ascending=False).head()

## Load the scan onto the genome

Windows colored by Fst — orange/red where the populations diverge. The peak sits over *LCT*.

In [ ]:
from jbrowse_anywidget import LinearGenomeView, make_assembly

grch38 = make_assembly(
    "GRCh38",
    "https://s3.amazonaws.com/jbrowse.org/genomes/GRCh38/fasta/GRCh38.fa.gz",
    aliases=["hg38"],
)
view = LinearGenomeView(assembly=grch38, location="2:135,000,000..137,000,000")
view.add_features(
    windows,
    name="Fst (pop1 vs pop2)",
    color="jexl:get(feature,'fst') > 0.25 ? '#d84315' : get(feature,'fst') > 0.1 ? '#f9a825' : '#90a4ae'",
)
view

Zoom to the *LCT* sweep:

In [ ]:
view.location = "2:135,700,000..135,900,000"